In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from unstructured_client import UnstructuredClient
from unstructured_client.models import shared
from unstructured_client.models.errors import SDKError

from unstructured.staging.base import dict_to_elements

import os
from dotenv import load_dotenv

In [3]:
load_dotenv()

api_key = os.getenv("UNSTRUCTURED_API_KEY")

In [4]:
from Utils import Utils
from unstructured_client import UnstructuredClient

utils = Utils()

API_KEY = utils.get_api_key()
API_URL = utils.get_url()

s = UnstructuredClient(
    api_key_auth=API_KEY,
    server_url=API_URL,
)

In [ ]:
filename = "example.pdf"

with open(filename, "rb") as f:
    files = shared.Files(
        content=f.read(),
        file_name=filename,
    )

req = shared.PartitionParameters(
    files=files,
    strategy="hi_res",
    hi_res_model_name="yolox",
    skip_infer_table_types=[],
    pdf_infer_table_structure=True,
)

try:
    resp = s.general.partition(
        request={
            "partition_parameters": req 
        }
    )
    elements = dict_to_elements(resp.elements)

    for el in elements[:5]:
        print(el.text)

except SDKError as e:
    print(e)

In [6]:
tables = [el for el in elements if el.category == "Table"]

In [7]:
tables[0].text

'Sample Date: May 2001 Prepared by: Accelio Present Applied Technology Created and Tested Using: • Accelio Present Central 5.4 • Accelio Present Output Designer 5.4 Features Demonstrated: • Primary bookmarks in a PDF file. • Secondary bookmarks in a PDF file.'

In [8]:
table_html = tables[0].metadata.text_as_html

In [9]:
from io import StringIO 
from lxml import etree

parser = etree.XMLParser(remove_blank_text=True)
file_obj = StringIO(table_html)
tree = etree.parse(file_obj, parser)
print(etree.tostring(tree, pretty_print=True).decode())

<table>
  <thead>
    <tr>
      <th>Sample Date:</th>
      <th>May 2001</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Prepared by:</td>
      <td>Accelio Present Applied Technology</td>
    </tr>
    <tr>
      <td>Created and Tested Using:</td>
      <td>) Accelio Present Central 5.4 Accelio Present Output Designer 5.4</td>
    </tr>
    <tr>
      <td>Features Demonstrated:</td>
      <td>Primary bookmarks in a PDF file. Secondary bookmarks in a PDF file.</td>
    </tr>
  </tbody>
</table>



In [10]:
from IPython.core.display import HTML
HTML(table_html)

Sample Date:,May 2001
Prepared by:,Accelio Present Applied Technology
Created and Tested Using:,) Accelio Present Central 5.4 Accelio Present Output Designer 5.4
Features Demonstrated:,Primary bookmarks in a PDF file. Secondary bookmarks in a PDF file.
